In [ ]:
import easyMPRA_20231120 as em
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd

In [ ]:

fq='cutadapt_FL_dict_S2_L002_R1_001.fastq'
enOrderSheet='ref/order-sheet-withControls_OLDNAMES.tsv'


In [ ]:
print(fq)
!wc -l {fq}

In [ ]:
out=f'01-Assemble-Dictionary/pe150-0-collapsed-reads.tsv'

read2count=\
    em.dict_step1_raw_to_collapsed_reads(rawReadFile=fq, 
                                         fileType='fastq',
                                         outfn=out, 
                                         readCol0idx=None)

In [ ]:
read2count=em.read2count_from_tsv(f'01-Assemble-Dictionary/pe150-0-collapsed-reads.tsv')

out=f'01-Assemble-Dictionary/pe150-locOnly-1-parsed-enhancers-barcodes.tsv'
readsWithEnhancerAndBarcode=\
em.dict_step2SR_parse_en_bc_from_read(enOrderSheet=enOrderSheet,
                                      read2count=read2count,
                                      bcLength=30,
                                      relPositionsToLookForLinker=[-3,-2,-1,+1,+2,+3],
                                      linkerSeq='CCTGCAGGGC',
                                      outfn=out,
                                      returnRevCompEn=True,
                                      returnRevCompBc=False,locationOnlyDoNotLookForLinker=True)

del read2count

In [ ]:
readsWithEnhancerAndBarcode=em.readsWithEnhancerAndBarcode_from_tsv(f'01-Assemble-Dictionary/pe150-locOnly-1-parsed-enhancers-barcodes.tsv')
# em.to_pickle(readsWithEnhancerAndBarcode,f'1-dict-processing/{seqrun}{locOnly}-1-parsed-enhancers-barcodes.readsWithEnhancerAndBarcode.pickle')

In [ ]:
out=f'01-Assemble-Dictionary/pe150-locOnly-2-raw-dict'
bc2en2count,missingEnhancers=\
em.dict_step3_assemble_dict(readsWithEnhancerAndBarcode=readsWithEnhancerAndBarcode,
                            enOrderSheet=enOrderSheet,
                            outDictBasename=out)

del readsWithEnhancerAndBarcode
del missingEnhancers

In [ ]:
bc2en2count=em.bc2en2count_from_tsv(f'01-Assemble-Dictionary/pe150-locOnly-2-raw-dict_bc2en2count.tsv_bc2en2count.tsv')

out=f'01-Assemble-Dictionary/3k-20240920-pe150-locOnly-3-filt_ubc2en_filt.minUReadsRequired=15.maxMmReadsAcceptable=15.bcTrimLength=25'
ubc2en,filtbc2en2count=\
em.dict_step4_filter_dictionary(bc2en2count,
                                out,
                                minUReadsRequired=15,
                                maxMmReadsAcceptable=15,
                                bcTrimLength=25          )